# The Finite Cosine: Multiplying in Space


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-16.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

A cosine that runs forever has a Fourier transform of two spikes: one frequency,
nothing else. Cut it off after a few cycles and those spikes have to broaden, because
a signal of finite length cannot contain a single exact frequency.

This notebook shows how much they broaden, and why. Cutting the cosine off is a
**multiplication** in space, by a rect window. The convolution theorem says a product
in one domain is a convolution in the other, so in frequency the two spikes get
convolved with the window's transform, and each spike becomes a sinc.

`ch05-15` runs this the other way: convolve in space, multiply in frequency. The two
notebooks are the same theorem read in opposite directions, which is why they use the
same layout.

## Import libraries

`ConnectionPatch` draws the arrows between subplots that turn six axes into one flow
diagram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch


In [ ]:
%matplotlib inline
# ==========================================
# 1. Configuration & Styling
# ==========================================
plot_params = {
    'font.size': 14,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Define parameters

**The window width $d$ and the cosine's frequency $k_0$** are the only physical
choices. $d = 6$ holds roughly four cycles of a $k_0 = 4$ cosine, which is few enough
that the broadening is obvious and many enough that the two peaks stay separate.

**The grid is deliberately long**, $L = 200$ against a window of 6. A Fourier transform
assumes the signal repeats forever, so leaving plenty of zero on either side keeps the
copies from interfering with each other.

**The transforms are analytic.** $\mathscr{F}\{\mathrm{rect}\}$ is a sinc from the
table, and the chapter derives $G(k)$ by hand, twice: once directly and once
through the convolution theorem, both giving

$$\mathscr{F}\left\{\mathrm{rect}\left(\frac{x}{d}\right)\cos(k_0 x)\right\}
= \frac{d}{2}\left\{\mathrm{sinc}\left[(k_0-k)\frac{d}{2}\right]
+ \mathrm{sinc}\left[(k_0+k)\frac{d}{2}\right]\right\}$$

which is what the bottom-right panel draws. Watch
the sinc convention: this text uses the **unnormalized** $\sin(u)/u$ while `np.sinc` is
the **normalized** $\sin(\pi z)/(\pi z)$, hence the helper that divides by $\pi$.

In [ ]:
# ==========================================
# 2. Data Generation
# ==========================================
width1 = 6.0      # d, the width of the window
k_zero1 = 4.0     # k_0, the spatial frequency of the cosine

N = 32768
L = 200.0
x = np.linspace(-L / 2, L / 2, N, endpoint=False)
dx = x[1] - x[0]

# Angular spatial frequency, this text's convention
k = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(N, d=dx))


def unnormalized_sinc(u):
    """sinc(u) = sin(u)/u, this text's convention.

    np.sinc is the NORMALIZED sin(pi z)/(pi z), so divide the argument by pi.
    """
    return np.sinc(u / np.pi)


# --- Space domain ---
f_val = np.cos(k_zero1 * x)                              # an endless cosine
h_val = np.where(np.abs(x) <= width1 / 2, 1.0, 0.0)      # the window
g_val = f_val * h_val                                    # their PRODUCT

# --- Frequency domain ---
# F{cos(k0 x)} = pi[delta(k - k0) + delta(k + k0)]: drawn as two arrows,
# since a delta has no finite height to plot.
delta_at1 = k_zero1

# F{rect(x/d)} = d sinc(kd/2)
H_val = width1 * unnormalized_sinc(k * width1 / 2)

# The chapter works this out by hand: a sinc centered on each delta.
G_val = (width1 / 2) * (unnormalized_sinc((k_zero1 - k) * width1 / 2)
                        + unnormalized_sinc((k_zero1 + k) * width1 / 2))


## Plot

The same 2x3 flow diagram as `ch05-15`, with the operations swapped.

The **top row multiplies**: an endless cosine, times a rect window, gives a cosine that
stops. Nothing subtle happens here, which is the point. The interesting consequence is
one row down.

The **bottom row convolves**. $F(k)$ is drawn as two arrows because a delta function has
no finite height to plot; the $\pi$ beside each marks its weight. Convolving anything
with a delta simply copies it to that delta's position, so convolving the sinc $H(k)$
with two deltas puts **one sinc at $+k_0$ and one at $-k_0$**. That is the whole result.

Read the bottom-right panel against the top-left. The infinite cosine had two infinitely
sharp spikes. Truncating it replaced each spike with a sinc of half-width $2\pi/d$. The
shorter the window, the wider each sinc: **a shorter measurement buys a blurrier
spectrum**, and that is a statement about Fourier transforms, not about any particular
instrument.

In [ ]:
# ==========================================
# 3. Plotting the Grid
# ==========================================
fig1, axes1 = plt.subplots(2, 3, figsize=(16, 9))
plt.subplots_adjust(wspace=0.50, hspace=0.78, left=0.13, right=0.97,
                    top=0.90, bottom=0.08)

xlim1 = 7.0
klim1 = 11.0

# --- Top Row: Space Domain ---
ax_f = axes1[0, 0]
ax_f.plot(x, f_val, 'r')
ax_f.set_title(r"$f(x) = \cos(k_0 x)$", fontsize=16, color='red')
ax_f.set_xlim(-xlim1, xlim1)
ax_f.set_ylim(-1.4, 1.4)
ax_f.set_xlabel(r"$x$")
apply_custom_formatting(ax_f)

ax_h = axes1[0, 1]
ax_h.plot(x, h_val, 'b')
ax_h.set_title(r"$h(x) = \mathrm{rect}(x/d)$", fontsize=16, color='blue')
ax_h.set_xlim(-xlim1, xlim1)
ax_h.set_ylim(-1.4, 1.4)
ax_h.set_xlabel(r"$x$")
apply_custom_formatting(ax_h)

ax_g = axes1[0, 2]
ax_g.plot(x, g_val, 'g')
ax_g.set_title(r"$g(x) = f \cdot h$   (a finite cosine)",
               fontsize=16, color='green')
ax_g.set_xlim(-xlim1, xlim1)
ax_g.set_ylim(-1.4, 1.4)
ax_g.set_xlabel(r"$x$")
ax_g.annotate("", xy=(-width1 / 2, -1.15), xytext=(width1 / 2, -1.15),
              arrowprops=dict(arrowstyle="<->", lw=1.5, color='green'))
ax_g.text(0, -1.05, r"$d$", ha='center', color='green', fontsize=14)
apply_custom_formatting(ax_g)

# --- Bottom Row: Frequency Domain ---
ax_F = axes1[1, 0]
# A delta cannot be plotted, so draw an arrow of unit height and label its weight
for sign in (+1, -1):
    ax_F.annotate("", xy=(sign * delta_at1, 1.0), xytext=(sign * delta_at1, 0),
                  arrowprops=dict(arrowstyle="-|>", lw=3, color='darkred'))
ax_F.text(delta_at1 + 0.5, 1.02, r"$\pi$", color='darkred', fontsize=15)
ax_F.text(-delta_at1 - 0.5, 1.02, r"$\pi$", color='darkred',
          fontsize=15, ha='right')
ax_F.set_title(r"$F(k) = \pi[\delta(k-k_0) + \delta(k+k_0)]$",
               fontsize=16, color='darkred')
ax_F.set_xlim(-klim1, klim1)
ax_F.set_ylim(-0.25, 1.45)
ax_F.set_xlabel(r"$k$")
apply_custom_formatting(ax_F)

ax_H = axes1[1, 1]
ax_H.plot(k, H_val, 'darkblue')
ax_H.set_title(r"$H(k) = d\,\mathrm{sinc}(kd/2)$",
               fontsize=16, color='darkblue')
ax_H.set_xlim(-klim1, klim1)
ax_H.set_xlabel(r"$k$")
apply_custom_formatting(ax_H)

ax_G = axes1[1, 2]
ax_G.plot(k, G_val, 'darkgreen')
ax_G.fill_between(k, 0, G_val, color='green', alpha=0.2)
for sign in (+1, -1):
    ax_G.axvline(sign * k_zero1, color='0.5', linestyle=':', linewidth=1.5)
ax_G.set_title(r"$G(k)$   (a sinc at each delta)", fontsize=16, color='darkgreen')
ax_G.set_xlim(-klim1, klim1)
ax_G.set_xlabel(r"$k$")
ax_G.text(k_zero1 + 0.4, width1 / 2 * 0.82, r"$k_0$", color='0.35', fontsize=14)
ax_G.text(-k_zero1 - 0.4, width1 / 2 * 0.82, r"$-k_0$", color='0.35',
          fontsize=14, ha='right')
apply_custom_formatting(ax_G)

# ==========================================
# 4. Adding Flowchart Annotations
# ==========================================
fig1.canvas.draw()


def draw_arrow(ax_from, ax_to, text=""):
    """Vertical arrow between a column's two panels, as in ch05-10."""
    con = ConnectionPatch(xyA=(0.5, -0.32), xyB=(0.5, 1.14),
                          coordsA="axes fraction", coordsB="axes fraction",
                          axesA=ax_from, axesB=ax_to,
                          arrowstyle="-|>", shrinkB=5, linewidth=2.5, color="black")
    ax_from.add_artist(con)
    if text:
        ax_to.text(0.56, 1.20, text, transform=ax_to.transAxes,
                   ha='left', va='bottom', fontsize=15)


draw_arrow(ax_f, ax_F, text="F.T.")
draw_arrow(ax_h, ax_H, text="F.T.")
draw_arrow(ax_g, ax_G, text="F.T.")


def gutter(ax_left, ax_right):
    """Center of the CLEAR gap between two side-by-side panels.

    The right panel's y-tick labels stick out to the left of its axes box, so
    centering between the boxes alone drops the glyph on top of them. Measuring
    where those labels actually begin gives a gap that is genuinely empty, and
    removes the need for a hand-tuned nudge.
    """
    fig = ax_left.figure
    renderer = fig.canvas.get_renderer()
    inv = fig.transFigure.inverted()
    right_x0 = ax_right.get_position().x0
    for lab in ax_right.get_yticklabels():
        if lab.get_visible() and lab.get_text():
            right_x0 = min(
                right_x0,
                lab.get_window_extent(renderer).transformed(inv).x0)
    return (ax_left.get_position().x1 + right_x0) / 2


def row_mid(ax):
    p = ax.get_position()
    return (p.y0 + p.y1) / 2


gx1, gx2 = gutter(ax_f, ax_h), gutter(ax_h, ax_g)
y_top, y_bot = row_mid(ax_f), row_mid(ax_F)

# One x for both the operator word and its symbol, so they stay aligned.
# gutter() already accounts for the tick labels, so no nudge is wanted here:
# the old one pushed the word onto the left panel's frame.
op_x = gx1


fig1.text(op_x, y_top + 0.055, "multiply", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_top, r"$\times$", fontsize=32, ha='center', va='center')
fig1.text(gx2, y_top, r"$=$", fontsize=32, ha='center', va='center')

fig1.text(op_x, y_bot + 0.055, "convolve", fontsize=12, ha='center', va='center')
fig1.text(op_x, y_bot, r"$\circledast$", fontsize=32,
          ha='center', va='center')
fig1.text(gx2, y_bot, r"$=$", fontsize=32, ha='center', va='center')

fig1.text(0.045, y_top, "Space\nDomain", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')
fig1.text(0.045, y_bot, "Frequency\nDomain", fontsize=17, weight='bold',
          rotation=90, va='center', ha='center')

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.

In [ ]:
# fig1.savefig("../figures/ch05/ch05-16-finite-cosine.jpg", dpi=200, bbox_inches='tight')


## The pair on its own

The grid above is the full story, but it is more than the chapter needs at the point
where this transform is first derived. There, the reader has just finished computing
$\mathscr{F}\{\mathrm{rect}(x/d)\cos(k_0 x)\}$ by direct integration and has not
yet met convolution.

So this second figure shows only the two things that exist at that stage: **the finite
cosine, and the transform just derived.** One function, one arrow, one result.

It is worth reading it for what it says on its own. The transform is two sincs, one
centered on $+k_0$ and one on $-k_0$, and each has a central lobe of width $4\pi/d$.
That width is the entire content of the result: **the shorter the window $d$, the wider
each sinc**, so a measurement of finite length cannot report a single exact frequency.
Truncating a wave in space blurs it in frequency, and the blur is inversely
proportional to how long you looked.

In [ ]:
# ==========================================
# 4. The pair on its own: the finite cosine and its transform
# ==========================================
# This is the result the chapter derives by hand, with nothing else attached.
fig2, (ax_sp2, ax_fq2) = plt.subplots(2, 1, figsize=(9, 8.5))
plt.subplots_adjust(hspace=0.68, left=0.13, right=0.95, top=0.93, bottom=0.09)

# --- The function ---
ax_sp2.plot(x, g_val, 'g')
ax_sp2.set_title(r"$g(x) = \mathrm{rect}(x/d)\,\cos(k_0 x)$",
                 fontsize=17, color='green')
ax_sp2.set_xlim(-xlim1, xlim1)
ax_sp2.set_ylim(-1.4, 1.4)
ax_sp2.set_xlabel(r"$x$")
ax_sp2.annotate("", xy=(-width1 / 2, -1.18), xytext=(width1 / 2, -1.18),
                arrowprops=dict(arrowstyle="<->", lw=1.5, color='green'))
ax_sp2.text(0, -1.09, r"$d$", ha='center', color='green', fontsize=15)
apply_custom_formatting(ax_sp2)

# --- Its transform ---
ax_fq2.plot(k, G_val, 'darkgreen')
ax_fq2.fill_between(k, 0, G_val, color='green', alpha=0.2)
for sign in (+1, -1):
    ax_fq2.axvline(sign * k_zero1, color='0.5', linestyle=':', linewidth=1.5)
ax_fq2.set_title(r"$G(k) = \frac{d}{2}\left\{\mathrm{sinc}"
                 r"\left[(k_0-k)\frac{d}{2}\right] + \mathrm{sinc}"
                 r"\left[(k_0+k)\frac{d}{2}\right]\right\}$",
                 fontsize=16, color='darkgreen')
ax_fq2.set_xlim(-klim1, klim1)
ax_fq2.set_xlabel(r"$k$")
ax_fq2.text(k_zero1 + 0.4, width1 / 2 * 0.86, r"$k_0$", color='0.35', fontsize=15)
ax_fq2.text(-k_zero1 - 0.4, width1 / 2 * 0.86, r"$-k_0$", color='0.35',
            fontsize=15, ha='right')

# The width of each sinc lobe is set by the window length
lobe2 = 2 * np.pi / width1
ax_fq2.annotate("", xy=(k_zero1 - lobe2, width1 / 2 * 0.32),
                xytext=(k_zero1 + lobe2, width1 / 2 * 0.32),
                arrowprops=dict(arrowstyle="<->", lw=1.5, color='darkgreen'))
ax_fq2.text(k_zero1, width1 / 2 * 0.38, r"$4\pi/d$", ha='center',
            color='darkgreen', fontsize=14)
apply_custom_formatting(ax_fq2)

# One arrow, because at this point in the chapter that is the only operation
# The arrow's length is the gap MINUS both offsets, so generous offsets leave
# nothing to draw. The lower one has to clear that panel's tall formula title,
# which is why hspace is roomy here.
con2 = ConnectionPatch(xyA=(0.5, -0.10), xyB=(0.5, 1.34),
                       coordsA="axes fraction", coordsB="axes fraction",
                       axesA=ax_sp2, axesB=ax_fq2,
                       arrowstyle="-|>", shrinkB=5, linewidth=2.5, color="black")
ax_sp2.add_artist(con2)
# Beside the arrow rather than above it, to stay clear of the lower title
ax_sp2.text(0.56, -0.30, "F.T.", transform=ax_sp2.transAxes,
            ha='left', va='center', fontsize=16)

plt.show()


### Save plot

Commented out on purpose, as above.

In [ ]:
# fig2.savefig("../figures/ch05/ch05-16-finite-cosine-pair.jpg", dpi=200, bbox_inches='tight')
